In [1]:
def input_embeddings(vocab_size, d_model):


_IncompleteInputError: incomplete input (2556812578.py, line 1)

In [3]:
import torch
import torch.nn as nn

In [33]:

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')

In [146]:
class InputEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model=d_model
        self.x = nn.Embedding(vocab_size,d_model).requires_grad_(True)
    def forward(self,z):
        out = self.x(z)/self.d_model**0.5
        return out

In [151]:
i = InputEmbeddings(8,4)
x = torch.arange(4)
i(x)

tensor([[ 0.2897, -0.9064,  0.0735, -0.2404],
        [-0.1841,  0.9068,  0.0818, -0.0889],
        [ 0.6722,  0.5545,  0.0075,  0.6314],
        [ 0.3250,  0.0989,  0.3800, -0.6604]], grad_fn=<DivBackward0>)

In [143]:
class PositionEncoding(nn.Module):
    def __init__(self, d_model:int, max_len:int, dropout:float):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # position
        position = torch.arange(1,max_len+1, dtype=torch.float16).unsqueeze(1)
        x = torch.sin(position/((10000**torch.tensor(torch.arange(0,d_model,2)/d_model))))
        y = torch.cos(position/((10000**torch.tensor(torch.arange(1,d_model,2)/d_model))))
        self.pos = torch.cat((x,y),dim=-1, device=device).unsqueeze(0)

    def forward(self,x):
        # self.pos => shape=> (1,max_len_d_model)
        return x+self.pos

In [150]:

position = torch.arange(1,max_len+1, dtype=torch.float16).unsqueeze(1)
mat = torch.zeros(max_len, d_model).to(device)
x = torch.sin(position/((10000**torch.tensor(torch.arange(0,d_model,2)/d_model))))
y = torch.cos(position/((10000**torch.tensor(torch.arange(1,d_model,2)/d_model))))
mat[:,::2] = x
mat[:,1::2] = y
pos = mat.unsqueeze(0)
pos.shape

/var/folders/tf/0hsnm4zx7w998xhnmj8tqn3c0000gn/T/ipykernel_21318/48145125.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.sin(position/((10000**torch.tensor(torch.arange(0,d_model,2)/d_model))))
/var/folders/tf/0hsnm4zx7w998xhnmj8tqn3c0000gn/T/ipykernel_21318/48145125.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.cos(position/((10000**torch.tensor(torch.arange(1,d_model,2)/d_model))))


torch.Size([1, 4, 10])

In [239]:
class LayerNorm(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, layer,eps):

        self.eps=eps
        self.layer = layer
        layer = self.layer
        m = torch.mean(layer, dim=-1, keepdim=True)
        var = torch.var(layer, dim=-1, keepdim=True)
        gamma = torch.ones(layer.shape[-2]).unsqueeze(1)
        beta = torch.zeros(layer.shape[-2]).unsqueeze(1)
        self.layerNorm = gamma*((layer-m)/((var+eps)**0.5))+ beta
        return self.layerNorm


In [325]:
class feedforward(nn.Module):
    def __init__(self, in_features,dff, dropout=0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.l = nn.Linear(in_features=in_features, out_features=dff)
        self.l2 = nn.Linear(in_features=dff, out_features=in_features)
        self.relu = nn.ReLU()
        # relu = nn.ReLU()
    
    def forward(self,x):
        return self.l2(self.dropout(self.relu(self.l(x))))


In [326]:
class MHA(nn.Module):
    def __init__(self,d_model):
        super().__init__()
        self.h = h
        self.d_model = d_model
        assert d_model % h==0
        self.w_q = torch.randn(d_model,d_model)
        self.w_k = torch.randn(d_model,d_model)
        self.w_v = torch.randn(d_model,d_model)

        self.w_o = torch.randn(d_model, d_model)

    def forward(self,q,k,v,mask=None):
        h  =self.h
        q_ = q@self.w_q
        k_ = k@self.w_k
        v_ = v@self.w_v # seq_len , d_model



        q = q_.view(q_.shape[0],-1,h, d_model//h).transpose(-2,-3) # batch, head, seq_len, dk
        k = k_.view(k_.shape[0],-1,h, d_model//h).transpose(-2,-3)
        v = v_.view(v_.shape[0],-1,h, d_model//h).transpose(-2,-3)

        x = nn.Softmax(dim=-1)
        z = (x(torch.tensor((q@(k.transpose(-2,-1)))/((d_model//h)**0.5)))@v).transpose(-2,-3)
        z = z.contiguous().view(z.shape[0],-1,d_model)
        z = z@self.w_o
        return z

In [327]:
class Residual(nn.Module):
    def __init__(self,dropout):
        self.dropout = nn.Dropout(dropout)
    def forward(self,x, fn=None):
        s = self.dropout(x)
        if fn is not None:
            return fn(x)
        return x



In [328]:
class EncoderBlock(nn.Module):
    def __init__(self,d_model,in_features, dff):
        super().__init__()
        self.MHA = MHA(d_model)
        # self.Residual = Residual()
        self.feedforward = feedforward(in_features, dff)
        self.layerNorm=LayerNorm()

    def forward(self,x,mask=None):
        # z = nn.ModuleList([self.Residual,self.Residual])
        x+=self.MHA(x,x,x)
        x = self.layerNorm(x,10e-5)
        x += self.feedforward(x)
        x = self.layerNorm(x,10e-5)
        return x






In [329]:
class Encoder(nn.Module):
    def __init__(self,N,EncoderBlock,d_model,in_features, dff):
        super().__init__()
        self.EncoderBlock=EncoderBlock(d_model,in_features, dff)
        self.N = N

    def forward(self,x):
        for _ in range(self.N):
            x = self.EncoderBlock(x)
        return x

In [333]:
d_model = 16
in_features = 16
dff=8
N=3
x = InputEmbeddings(10,16)
y = torch.arange(5)
z = x(y).unsqueeze(0)
# z = z+
Encoder_ = Encoder(N,EncoderBlock,d_model,in_features, dff)
Encoder_(z).shape, z.shape

/var/folders/tf/0hsnm4zx7w998xhnmj8tqn3c0000gn/T/ipykernel_21318/3991622472.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  z = (x(torch.tensor((q@(k.transpose(-2,-1)))/((d_model//h)**0.5)))@v).transpose(-2,-3)


(torch.Size([1, 5, 16]), torch.Size([1, 5, 16]))

In [294]:

x = InputEmbeddings(10,16)
y = torch.arange(5)
x(y).shape

torch.Size([5, 16])